# Additional End of week Exercise - week 3

Exercise: A business challenge 

Generating synthetic data 
- write models that can generate datasets
- use a variety of models (open source-ollama and closed source-openai) and prompts for diverse outputs 
- create a Gradio UI for your product 

In [11]:
import os
import json
import re
import tempfile
import pandas as pd
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI/OpenRouter API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - required for GPT-4.1-mini via OpenRouter")

OpenAI/OpenRouter API Key exists and begins sk-or-v1


In [12]:
openai = OpenAI(base_url="https://openrouter.ai/api/v1")
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

MODELS = {
    "GPT-4.1-mini (OpenRouter)": {
        "client": openai,
        "model_id": "gpt-4.1-mini",
        "type": "closed-source",
    },
    "Llama 3.2 1B (Ollama)": {
        "client": ollama,
        "model_id": "llama3.2:1b",
        "type": "open-source",
    },
    "DeepSeek-R1 1.5B (Ollama)": {
        "client": ollama,
        "model_id": "deepseek-r1:1.5b",
        "type": "open-source",
    },
}

print("Models registered:")
for name, info in MODELS.items():
    print(f"  - {name} ({info['type']})")

Models registered:
  - GPT-4.1-mini (OpenRouter) (closed-source)
  - Llama 3.2 1B (Ollama) (open-source)
  - DeepSeek-R1 1.5B (Ollama) (open-source)


In [13]:
TICKET_CATEGORIES = [
    "Billing",
    "Technical Support",
    "Account Access",
    "Feature Request",
    "Bug Report",
    "General Inquiry",
]

PRIORITIES = ["Low", "Medium", "High", "Critical"]

SENTIMENTS = ["Positive", "Neutral", "Negative", "Frustrated"]

TICKET_SCHEMA = {
    "ticket_id": "Unique integer starting from 1",
    "customer_name": "Realistic full name",
    "email": "Matching email address",
    "category": f"One of: {', '.join(TICKET_CATEGORIES)}",
    "priority": f"One of: {', '.join(PRIORITIES)}",
    "subject": "Short ticket subject line",
    "description": "2-4 sentence customer message describing their issue",
    "sentiment": f"One of: {', '.join(SENTIMENTS)}",
}

print("Ticket schema fields:")
for field, desc in TICKET_SCHEMA.items():
    print(f"  {field}: {desc}")

Ticket schema fields:
  ticket_id: Unique integer starting from 1
  customer_name: Realistic full name
  email: Matching email address
  category: One of: Billing, Technical Support, Account Access, Feature Request, Bug Report, General Inquiry
  priority: One of: Low, Medium, High, Critical
  subject: Short ticket subject line
  description: 2-4 sentence customer message describing their issue
  sentiment: One of: Positive, Neutral, Negative, Frustrated


In [14]:
SCHEMA_BLOCK = json.dumps(TICKET_SCHEMA, indent=2)


def strategy_direct(num_records, categories):
    """Direct/Structured — explicit JSON schema, strict instructions."""
    cats = ", ".join(categories)
    system = (
        "You are a synthetic data generator for CloudSync, a SaaS company. "
        "You output ONLY valid JSON arrays — no markdown, no explanation."
    )
    user = (
        f"Generate exactly {num_records} realistic customer support tickets as a JSON array.\n\n"
        f"Each object must have these fields:\n{SCHEMA_BLOCK}\n\n"
        f"Use only these categories: {cats}\n"
        "Vary the names, issues, priorities, and sentiments realistically. "
        "Return ONLY the JSON array."
    )
    return system, user


def strategy_few_shot(num_records, categories):
    """Few-Shot — provide example records in the prompt."""
    cats = ", ".join(categories)
    examples = json.dumps([
        {
            "ticket_id": 1,
            "customer_name": "Maria Chen",
            "email": "maria.chen@example.com",
            "category": "Billing",
            "priority": "High",
            "subject": "Double charged for monthly subscription",
            "description": "I was charged twice on my credit card for my CloudSync Pro subscription this month. The duplicate charge of $29.99 appeared on March 1st. Please refund the extra charge.",
            "sentiment": "Frustrated",
        },
        {
            "ticket_id": 2,
            "customer_name": "James Okonkwo",
            "email": "j.okonkwo@techcorp.io",
            "category": "Technical Support",
            "priority": "Medium",
            "subject": "File sync stuck at 85%",
            "description": "My CloudSync desktop client has been stuck syncing at 85% for the past 3 hours. I've tried restarting the app and my computer but the issue persists. Running macOS 14.2.",
            "sentiment": "Negative",
        },
    ], indent=2)
    system = (
        "You are a synthetic data generator for CloudSync, a SaaS company. "
        "You output ONLY valid JSON arrays — no markdown, no explanation."
    )
    user = (
        f"Here are example customer support tickets:\n{examples}\n\n"
        f"Now generate exactly {num_records} NEW and DIFFERENT tickets in the same JSON format.\n"
        f"Use only these categories: {cats}\n"
        "Make each ticket unique with different names, issues, and tones. "
        "Start ticket_id from 1. Return ONLY the JSON array."
    )
    return system, user


def strategy_persona(num_records, categories):
    """Persona-Based — model role-plays as different customer types."""
    cats = ", ".join(categories)
    system = (
        "You are simulating real customers of CloudSync, a SaaS file-sync platform. "
        "You will role-play as different customer personas and write support tickets "
        "from their perspectives. Output ONLY a valid JSON array — no markdown, no explanation."
    )
    user = (
        f"Generate exactly {num_records} support tickets as a JSON array.\n\n"
        f"Each ticket must follow this schema:\n{SCHEMA_BLOCK}\n\n"
        f"Use only these categories: {cats}\n\n"
        "Rotate through these customer personas:\n"
        "1. Frustrated power-user: technically savvy, impatient, uses jargon\n"
        "2. Confused beginner: new to the product, unsure how things work\n"
        "3. Friendly long-time customer: loyal, polite, but has a real issue\n"
        "4. Angry customer threatening to cancel: upset, demands escalation\n\n"
        "Make each ticket feel authentic to its persona. Start ticket_id from 1. "
        "Return ONLY the JSON array."
    )
    return system, user


def strategy_scenario(num_records, categories):
    """Scenario-Driven — tickets around specific business events."""
    cats = ", ".join(categories)
    system = (
        "You are a synthetic data generator for CloudSync, a SaaS company. "
        "You output ONLY valid JSON arrays — no markdown, no explanation."
    )
    user = (
        f"CloudSync recently experienced several events:\n"
        "- A 4-hour service outage last Tuesday affecting file sync\n"
        "- A new 'Enterprise Plus' pricing tier was introduced\n"
        "- Two-factor authentication was made mandatory for all accounts\n"
        "- The mobile app was updated to v3.0 with a redesigned UI\n\n"
        f"Generate exactly {num_records} support tickets as a JSON array, "
        "where each ticket relates to one of these events.\n\n"
        f"Each object must have these fields:\n{SCHEMA_BLOCK}\n\n"
        f"Use only these categories: {cats}\n"
        "Start ticket_id from 1. Return ONLY the JSON array."
    )
    return system, user


PROMPT_STRATEGIES = {
    "Direct/Structured": strategy_direct,
    "Few-Shot": strategy_few_shot,
    "Persona-Based": strategy_persona,
    "Scenario-Driven": strategy_scenario,
}

print(f"Prompt strategies available: {list(PROMPT_STRATEGIES.keys())}")

Prompt strategies available: ['Direct/Structured', 'Few-Shot', 'Persona-Based', 'Scenario-Driven']


In [15]:
def parse_json_response(text):
    """Parse JSON from model response, handling markdown fences and DeepSeek <think> tags."""
    # Strip DeepSeek <think>...</think> reasoning blocks
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)

    # Strip markdown code fences
    text = re.sub(r"```(?:json)?\s*", "", text)
    text = re.sub(r"```", "", text)
    text = text.strip()

    # Try direct parse first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback: extract the first JSON array from the text
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    return None


def generate_tickets(model_name, strategy_name, num_records, categories):
    """Call a model with a prompt strategy and return (DataFrame, status_message)."""
    model_info = MODELS[model_name]
    strategy_fn = PROMPT_STRATEGIES[strategy_name]

    system_msg, user_msg = strategy_fn(num_records, categories)

    try:
        response = model_info["client"].chat.completions.create(
            model=model_info["model_id"],
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg},
            ],
            temperature=0.9,
        )
        raw = response.choices[0].message.content
    except Exception as e:
        return pd.DataFrame(), f"ERROR calling {model_name}: {e}"

    tickets = parse_json_response(raw)
    if tickets is None:
        return pd.DataFrame(), f"ERROR: Could not parse JSON from {model_name}. Raw output:\n{raw[:500]}"

    if isinstance(tickets, dict):
        tickets = [tickets]

    df = pd.DataFrame(tickets)
    df["model"] = model_name
    df["strategy"] = strategy_name

    return df, f"{model_name} + {strategy_name}: generated {len(df)} tickets"


print("Generation engine ready.")

Generation engine ready.


In [16]:
test_df, test_status = generate_tickets(
    model_name="GPT-4.1-mini (OpenRouter)",
    strategy_name="Direct/Structured",
    num_records=3,
    categories=TICKET_CATEGORIES,
)

print(test_status)
test_df

GPT-4.1-mini (OpenRouter) + Direct/Structured: generated 3 tickets


,ticket_id,customer_name,email,category,priority,subject,description,sentiment,model,strategy
0,1,Maria Gonzalez,maria.gonzalez89@example.com,Technical Support,High,Sync process failing intermittently,The syncing process keeps failing randomly thr...,Frustrated,GPT-4.1-mini (OpenRouter),Direct/Structured
1,2,James O'Connor,james.oconnor@example.com,Billing,Medium,Incorrect charge on last invoice,I noticed an extra charge on my latest invoice...,Neutral,GPT-4.1-mini (OpenRouter),Direct/Structured
2,3,Anita Shah,anita.shah@example.com,Feature Request,Low,Request for dark mode option,It would be great if CloudSync offered a dark ...,Positive,GPT-4.1-mini (OpenRouter),Direct/Structured


In [17]:
def generate_batch(selected_models, strategy_name, num_records, categories, progress=gr.Progress()):
    """Generate tickets from multiple models, combine into one DataFrame."""
    all_dfs = []
    status_lines = []

    for i, model_name in enumerate(selected_models):
        progress((i) / len(selected_models), desc=f"Generating with {model_name}...")
        df, status = generate_tickets(model_name, strategy_name, num_records, categories)
        status_lines.append(status)
        if not df.empty:
            all_dfs.append(df)

    progress(1.0, desc="Done!")

    if not all_dfs:
        return pd.DataFrame(), "\n".join(status_lines)

    combined = pd.concat(all_dfs, ignore_index=True)
    # Re-number ticket IDs sequentially
    combined["ticket_id"] = range(1, len(combined) + 1)

    summary = "\n".join(status_lines)
    summary += f"\n\nTotal: {len(combined)} tickets from {len(selected_models)} model(s)"
    return combined, summary


print("Batch generation ready.")

Batch generation ready.


In [18]:
def export_csv(df):
    """Save DataFrame to a temp CSV and return the file path."""
    if df is None or df.empty:
        return None
    path = os.path.join(tempfile.gettempdir(), "cloudsync_tickets.csv")
    df.to_csv(path, index=False)
    return path


def export_json(df):
    """Save DataFrame to a temp JSON and return the file path."""
    if df is None or df.empty:
        return None
    path = os.path.join(tempfile.gettempdir(), "cloudsync_tickets.json")
    df.to_json(path, orient="records", indent=2)
    return path


print("Export helpers ready.")

Export helpers ready.


In [19]:
with gr.Blocks(title="CloudSync Synthetic Ticket Generator") as demo:
    # Store state as list of dicts (not DataFrame) to avoid Gradio bool() error on pandas
    latest_records = gr.State([])

    gr.Markdown("# CloudSync — Synthetic Support Ticket Generator")
    gr.Markdown(
        "Generate realistic customer support tickets using multiple AI models "
        "and diverse prompt strategies. Built for Week 3 exercise."
    )

    with gr.Row():
        # --- Left column: Controls ---
        with gr.Column(scale=1):
            model_select = gr.CheckboxGroup(
                choices=list(MODELS.keys()),
                value=list(MODELS.keys()),
                label="Select Models",
            )
            strategy_select = gr.Dropdown(
                choices=list(PROMPT_STRATEGIES.keys()),
                value="Direct/Structured",
                label="Prompt Strategy",
            )
            num_records = gr.Slider(
                minimum=1, maximum=25, value=5, step=1,
                label="Records per Model",
            )
            category_filter = gr.CheckboxGroup(
                choices=TICKET_CATEGORIES,
                value=TICKET_CATEGORIES,
                label="Categories to Include",
            )
            generate_btn = gr.Button("Generate Tickets", variant="primary")

        # --- Right column: Output ---
        with gr.Column(scale=2):
            status_box = gr.Textbox(label="Status", lines=6, interactive=False)
            output_table = gr.Dataframe(
                label="Generated Tickets",
                wrap=True,
                interactive=False,
            )
            with gr.Row():
                csv_btn = gr.Button("Download CSV")
                json_btn = gr.Button("Download JSON")
            file_output = gr.File(label="Download")

    # --- Callbacks ---
    def on_generate(models, strategy, n_records, categories):
        if not models:
            return pd.DataFrame(), "Please select at least one model.", []
        if not categories:
            return pd.DataFrame(), "Please select at least one category.", []
        df, status = generate_batch(models, strategy, int(n_records), categories)
        records = df.to_dict("records") if not df.empty else []
        return df, status, records

    def on_export_csv(records):
        if not records:
            return None
        return export_csv(pd.DataFrame(records))

    def on_export_json(records):
        if not records:
            return None
        return export_json(pd.DataFrame(records))

    generate_btn.click(
        fn=on_generate,
        inputs=[model_select, strategy_select, num_records, category_filter],
        outputs=[output_table, status_box, latest_records],
    )

    csv_btn.click(fn=on_export_csv, inputs=[latest_records], outputs=[file_output])
    json_btn.click(fn=on_export_json, inputs=[latest_records], outputs=[file_output])

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.
